In [3]:
import pandas as pd
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import shutil

In [4]:
def draw_event_bars(files,title,fig_path,is_save = False):
#draw vars graph of upp,lower,neutral neuron activities on multiple events

    event_params = []
    for file in files:
        temp = pd.read_csv(file)
        upper = temp['upper'].sum()
        lower = temp['lower'].sum()
        neutral = temp.index.size - upper - lower
        event_params.append([upper, lower, neutral])
    
    #fig parameters
    bar_width = 0.5  
    x = np.arange(1,len(event_params)+1)  
    event_params = np.array(event_params)

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.bar(x, event_params[:,2], width=bar_width, label='Neutral', color='antiquewhite')
    ax.bar(x,event_params[:,1], bottom=event_params[:,2], width=bar_width, label='In\hibition', color='darkred')
    ax.bar(x, event_params[:,0], bottom=event_params[:,1]+event_params[:,2], width=bar_width, label='Excitation', color='darkgreen')
    ax.set_title(title)
    ax.set_xticks(x)  
    ax.set_xticklabels(x) 
    #ax.legend()

    if is_save:
        plt.savefig(fig_path + 'cells_event_bars.pdf')
    plt.show()
    plt.close()
    return event_params

English note: release cleanup annotation.


In [ ]:
project_path = '${PROJECT_ROOT}'
test = 'test1'

# test1 params
cellType = 'CamkII_TST'#VIP/CamkII_TST/SST/PV/Hsyn

#test2 params
condition = 'rescue'#pre,post,rescue
experiment = 'TST'
mice_IDs = ['list of micIDs']
depressed = ['list of micIDs']
resist = ['list of micIDs']

match test:
    case 'test2':
        output_path = os.path.join(project_path,'summary',test,condition)
    case 'test1':
        output_path = os.path.join(project_path,'summary',test,cellType)



In [ ]:
#getting combined data
all_CellType_Data = []
match test:
    case 'test1':
        for s in range(1,11):
            subject_data = os.path.join(project_path,'summary',test,cellType,str(s),'signal_save','analysis_total.csv')
            if not os.path.exists(subject_data):
                print(f"File not found: {subject_data}")
                continue
            subject_df = pd.read_csv(subject_data)  # Display the first few rows of the DataFrame
            subject_df.loc[0,'Subject'] = cellType + str(s)
            all_CellType_Data.append(subject_df)
    case 'test2':
        for mouse_id in mice_IDs:
            subject_data = os.path.join(project_path,'summary',test,condition,mouse_id,experiment,'signal_save','analysis_total.csv')
            if not os.path.exists(subject_data):
                print(f"File not found: {subject_data}")
                continue
            subject_df = pd.read_csv(subject_data)  # Display the first few rows of the DataFrame
            subject_df.loc[0,'Subject'] = mouse_id
            all_CellType_Data.append(subject_df)

combined_data = pd.concat(all_CellType_Data, axis=0, ignore_index=True)
print(combined_data)
combined_data.to_csv(os.path.join(output_path,'Combined_Event.csv'), index=False)


In [ ]:
# analysis
#test1 looper
all_CellType_Freq = []
for s in range(1,2):
    if test == 'test2':#test2 not use this looper
        print("test2 mode, skip this looper.")
        break
    subject_dir = os.path.join(project_path,'summary',test,cellType,str(s))
    subject_data = os.path.join(subject_dir,'signal_save','analysis_total.csv')
    if not os.path.exists(subject_data):
        print(f"File not found: {subject_data}")
        continue
    subject_df = pd.read_csv(subject_data)
    subject_Freq_df = subject_df.copy()
    signal_folder = os.path.dirname(subject_data)
    if not os.path.exists(signal_folder):
        print(f"Folder not found: {signal_folder}")
        continue

     # SIanalysis
     # SI
    SIpattern = os.path.join(signal_folder, 'S_to_I_zscore_event_*.csv')
    matching_files = glob.glob(SIpattern)
    SI_num = len(matching_files)
    bar_path = os.path.join(subject_dir,'cell_event_bars')
    if not os.path.exists(bar_path):os.makedirs(bar_path)

     
    if matching_files:
        bar_data = draw_event_bars(matching_files,f'{cellType}{str(s)} S to I events',os.path.join(bar_path,'S_to_I_'),True)
        bar_data_df = pd.DataFrame(bar_data, columns=['S_to_I_up','S_to_I_down','S_to_I_neutral'])
        bar_data_df.to_csv(os.path.join(bar_path,'S_to_I_event_bars.csv'), index=False)

     # SI
    SIpattern = os.path.join(signal_folder, 'S_to_I_zscore_event_*.csv')
    matching_files = glob.glob(SIpattern)
    SI_num = len(matching_files)
    if matching_files:
         subject_Freq_df[['S_to_I_up','S_to_I_down']] /= SI_num
    else:
         subject_Freq_df[['S_to_I_up','S_to_I_down']] = 0


    # ISanalysis
    # IS
    ISpattern = os.path.join(signal_folder, 'I_to_S_zscore_event_*.csv')
    matching_files = glob.glob(ISpattern)
    IS_num = len(matching_files)
    if matching_files:
        bar_data = draw_event_bars(matching_files,f'{cellType}{str(s)} I to S events',os.path.join(bar_path,'I_to_S_'),True)
        bar_data_df = pd.DataFrame(bar_data, columns=['I_to_S_up','I_to_S_down','I_to_S_neutral'])
        bar_data_df.to_csv(os.path.join(bar_path,'I_to_S_event_bars.csv'), index=False)

     # IS
    if matching_files:
         subject_Freq_df[['I_to_S_up','I_to_S_down']] /= IS_num
    else:
         subject_Freq_df[['I_to_S_up','I_to_S_down']] = 0

    subject_Freq_df['Space'] = None
    #SI,IS, up&down freq
    event_num = SI_num + IS_num
    subject_Freq_df['SI up bias'] = (subject_df['S_to_I_up'] - subject_df['I_to_S_up']) / event_num if event_num > 0 else 0
    subject_Freq_df['SI down bias'] = (subject_df['S_to_I_down'] - subject_df['I_to_S_down']) / event_num if event_num > 0 else 0
    subject_Freq_df = subject_Freq_df.astype('float')

    subject_Freq_df.loc[0,'Subject'] = cellType + str(s)
    all_CellType_Freq.append(subject_Freq_df)

if all_CellType_Freq:
     combined_Freq = pd.concat(all_CellType_Freq, axis=0, ignore_index=True)
else:
     print("No frequency data to combine.")
     combined_Freq = pd.DataFrame()  # Create an empty DataFrame if no data
print(combined_Freq)
#combined_Freq.to_csv(os.path.join(output_path,'Combined_Event_Freq.csv'), index=False)


In [ ]:
# analysis
#test2 looper
all_CellType_Freq = []
for mice in mice_IDs:
    if test == 'test1':#test2 not use this looper
        print("test1 mode, skip this looper.")
        break
    subject_dir = os.path.join(project_path,'summary',test,condition,mice)
    subject_data = os.path.join(subject_dir,experiment,'signal_save','analysis_total.csv')
    if not os.path.exists(subject_data):
        print(f"Subject not found: {subject_data}")
        continue
    subject_df = pd.read_csv(subject_data)
    subject_Freq_df = subject_df.copy()
    signal_folder = os.path.dirname(subject_data)
    if not os.path.exists(signal_folder):
        print(f"Folder not found: {signal_folder}")
        continue

     # SIanalysis
     # SI
    SIpattern = os.path.join(signal_folder, 'S_to_I_zscore_event_*.csv')
    matching_files = glob.glob(SIpattern)
    SI_num = len(matching_files)
    old_bar_path = os.path.join(subject_dir,'cell_event_bars')
    if os.path.exists(old_bar_path):
        shutil.rmtree(old_bar_path)
    bar_path = os.path.join(subject_dir,experiment,'cell_event_bars')
    if not os.path.exists(bar_path):os.makedirs(bar_path)

     
    if matching_files:
        draw_event_bars(matching_files,f'{mice}{condition} S to I events',os.path.join(bar_path,'S_to_I_'),True)

     # SI
    SIpattern = os.path.join(signal_folder, 'S_to_I_zscore_event_*.csv')
    matching_files = glob.glob(SIpattern)
    SI_num = len(matching_files)
    if matching_files:
         subject_Freq_df[['S_to_I_up','S_to_I_down']] /= SI_num
    else:
         subject_Freq_df[['S_to_I_up','S_to_I_down']] = 0


    # ISanalysis
    # IS
    ISpattern = os.path.join(signal_folder, 'I_to_S_zscore_event_*.csv')
    matching_files = glob.glob(ISpattern)
    IS_num = len(matching_files)
    if matching_files:
        draw_event_bars(matching_files,f'{mice}{condition} I to S events',os.path.join(bar_path,'I_to_S_'),True)

     # IS
    if matching_files:
         subject_Freq_df[['I_to_S_up','I_to_S_down']] /= IS_num
    else:
         subject_Freq_df[['I_to_S_up','I_to_S_down']] = 0

    subject_Freq_df['Space'] = None
    #SI,IS, up&down freq
    event_num = SI_num + IS_num
    subject_Freq_df['SI up bias'] = (subject_df['S_to_I_up'] - subject_df['I_to_S_up']) / event_num if event_num > 0 else 0
    subject_Freq_df['SI down bias'] = (subject_df['S_to_I_down'] - subject_df['I_to_S_down']) / event_num if event_num > 0 else 0
    subject_Freq_df = subject_Freq_df.astype('float')

    subject_Freq_df.loc[0,'Subject'] = mice + '_' + condition
    all_CellType_Freq.append(subject_Freq_df)

if all_CellType_Freq:
     combined_Freq = pd.concat(all_CellType_Freq, axis=0, ignore_index=True)
else:
     print("No frequency data to combine.")
     combined_Freq = pd.DataFrame()  # Create an empty DataFrame if no data
print(combined_Freq)
combined_Freq.to_csv(os.path.join(output_path,'Combined_Event_Freq.csv'), index=False)
